In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
train_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/mohs_hardness/train.csv')

# Display the first few rows of the dataset
print(train_data.head())

# Get a summary of the dataset
print(train_data.info())

# Check for missing values
print(train_data.isnull().sum())

# Check for duplicate rows
print(train_data.duplicated().sum())

# Distinguish column types
numeric_cols = train_data.select_dtypes(include=[np.number]).columns
categorical_cols = train_data.select_dtypes(include=['object', 'category']).columns

print("Numeric Columns:", numeric_cols)
print("Categorical Columns:", categorical_cols)

# Visualize the distribution of numeric columns
for col in numeric_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(train_data[col], bins=30, kde=True)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.show()

# Visualize the distribution of categorical columns
for col in categorical_cols:
    plt.figure(figsize=(10, 4))
    sns.countplot(y=train_data[col], order=train_data[col].value_counts().index)
    plt.title(f'Distribution of {col}')
    plt.xlabel('Count')
    plt.ylabel(col)
    plt.show()

# Check for correlations among numeric columns
plt.figure(figsize=(12, 8))
correlation_matrix = train_data[numeric_cols].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Numeric Columns')
plt.show()


     id  allelectrons_Total  ...  density_Average  Hardness
0  2124                30.0  ...          0.51006       6.0
1   394                64.0  ...          4.74000       3.3
2  3101                97.0  ...          1.79976       5.3
3  1737               151.0  ...          7.77500       1.8
4   561               131.0  ...          1.92652       5.5

[5 rows x 13 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8325 entries, 0 to 8324
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     8325 non-null   int64  
 1   allelectrons_Total     8325 non-null   float64
 2   density_Total          8325 non-null   float64
 3   allelectrons_Average   8325 non-null   float64
 4   val_e_Average          8325 non-null   float64
 5   atomicweight_Average   8325 non-null   float64
 6   ionenergy_Average      8325 non-null   float64
 7   el_neg_chi_Average     8325 non-null 

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_data)
print("column_info")
print(column_info)


2025-09-15 00:22:08.793 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': [], 'Numeric': ['id', 'allelectrons_Total', 'density_Total', 'allelectrons_Average', 'val_e_Average', 'atomicweight_Average', 'ionenergy_Average', 'el_neg_chi_Average', 'R_vdw_element_Average', 'R_cov_element_Average', 'zaratio_Average', 'density_Average', 'Hardness'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import FillMissingValue, StandardScale

# Load the test data
test_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/mohs_hardness/test.csv')

# Copy the data to avoid modifying the original data
train_data_copy = train_data.copy()
test_data_copy = test_data.copy()

# Handle missing values
numeric_cols = train_data_copy.select_dtypes(include=[np.number]).columns
fill_missing = FillMissingValue(features=numeric_cols, strategy='mean')
train_data_copy = fill_missing.fit_transform(train_data_copy)
test_data_copy = fill_missing.transform(test_data_copy)

# Scale numerical features
scale = StandardScale(features=numeric_cols)
train_data_copy = scale.fit_transform(train_data_copy)
test_data_copy = scale.transform(test_data_copy)

# Display the first few rows of the processed data
print(train_data_copy.head())
print(test_data_copy.head())


         id  allelectrons_Total  ...  density_Average  Hardness
0 -1.028644           -0.449761  ...        -0.836903  0.799538
1 -1.604442           -0.293378  ...         1.334983 -0.799716
2 -0.703468           -0.141595  ...        -0.174700  0.384917
3 -1.157450            0.106778  ...         2.893320 -1.688190
4 -1.548859            0.014788  ...        -0.109614  0.503380

[5 rows x 13 columns]
         id  allelectrons_Total  ...  density_Average  Hardness
0 -0.420229           -0.219786  ...        -0.653507  0.503380
1 -0.205553            0.511535  ...        -0.141299  1.391854
2  1.466589            0.111378  ...        -0.835768 -1.273569
3  1.133425            0.479338  ...        -0.791950  1.273391
4 -0.306401           -0.376169  ...         0.960617 -1.273569

[5 rows x 13 columns]


In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_data_copy)
print("column_info")
print(column_info)


column_info
{'Category': [], 'Numeric': ['id', 'allelectrons_Total', 'density_Total', 'allelectrons_Average', 'val_e_Average', 'atomicweight_Average', 'ionenergy_Average', 'el_neg_chi_Average', 'R_vdw_element_Average', 'R_cov_element_Average', 'zaratio_Average', 'density_Average', 'Hardness'], 'Datetime': [], 'Others': []}


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

# Assuming train_data_copy and test_data_copy are already loaded and preprocessed
X_train = train_data_copy.drop(columns=['Hardness'])
y_train = train_data_copy['Hardness']
X_test = test_data_copy.drop(columns=['Hardness'])
y_test = test_data_copy['Hardness']

# Initialize and fit the XGBoost model
model = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=42)
model.fit(X_train, y_train, early_stopping_rounds=50, eval_set=[(X_test, y_test)], verbose=False)

# Predict on the test set
y_pred = model.predict(X_test)

# Calculate the Median Absolute Error (MAE)
mae = mean_absolute_error(y_test, y_pred)
print(f'Median Absolute Error: {mae}')


TypeError: XGBModel.fit() got an unexpected keyword argument 'early_stopping_rounds'